# Connexió a MongoDB Atlas
Aquest notebook comprova la connexió, llista col·leccions i compta documents.

In [4]:
import os
from pprint import pprint
from pymongo import MongoClient
from dotenv import load_dotenv

# Carregar variables d'entorn des de .env
load_dotenv()
MONGO_URI = os.getenv('MONGO_URI')
MONGO_DB = os.getenv('MONGO_DB', 'feb_db')

assert MONGO_URI, 'Falta MONGO_URI al .env'

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

# Fallback automàtic si la BD està buida
if len(db.list_collection_names()) == 0 and 'FEB_Basketball' in client.list_database_names():
    print(f"BD '{MONGO_DB}' buida. Fent fallback a 'FEB_Basketball'.")
    MONGO_DB = 'FEB_Basketball'
    db = client[MONGO_DB]

client.admin.command('ping')
print('Connexió OK a', MONGO_DB)

BD 'feb_db' buida. Fent fallback a 'FEB_Basketball'.
Connexió OK a FEB_Basketball


In [3]:
# Diagnòstic: llistar bases de dades i col·leccions
dbs = client.list_database_names()
print("Bases de dades disponibles:", dbs)

# Mostrar col·leccions per BD (només si n'hi ha poques per evitar massa sortida)
for db_name in dbs:
    try:
        col_names = client[db_name].list_collection_names()
        print(f"{db_name}: {len(col_names)} col·leccions")
    except Exception as e:
        print(f"{db_name}: error al llistar col·leccions ({e})")

Bases de dades disponibles: ['FEB_Basketball', 'admin', 'local']
FEB_Basketball: 1 col·leccions
admin: 0 col·leccions
local: 1 col·leccions


In [5]:
cols = db.list_collection_names()
print('Col·leccions trobades:', len(cols))
pprint(cols)

Col·leccions trobades: 1
['FEB3_players_statistics']


In [6]:
# Comptar documents per a les col·leccions clau
target_cols = ["partits", "FEB3_players_shots", "FEB3_players_statistics", "FEB3_teams_statistics"]
for name in target_cols:
    if name in cols:
        print(f'{name}: {db[name].estimated_document_count():,} documents')
    else:
        print(f'{name}: no existeix a la BD')

partits: no existeix a la BD
FEB3_players_shots: no existeix a la BD
FEB3_players_statistics: 97 documents
FEB3_teams_statistics: no existeix a la BD


# PART 1.1: Consultes de prova i filtres per temporada/competició

En aquesta secció completem les consultes de prova (samples, claus, comptatges) i afegim filtres per temporada i competició per a les col·leccions clau: `partits`, `FEB3_players_shots`, `FEB3_players_statistics`, `FEB3_teams_statistics`.

In [8]:
# Helper: garantir `db` i col·leccions clau
try:
    db
except NameError:
    import os
    from pymongo import MongoClient
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    uri = os.getenv("MONGO_URI")
    db_name = os.getenv("MONGO_DB")
    client = MongoClient(uri)
    db = client[db_name]

key_cols = [
    "partits",
    "FEB3_players_shots",
    "FEB3_players_statistics",
    "FEB3_teams_statistics",
]
found_cols = [c for c in key_cols if c in db.list_collection_names()]
print("Col·leccions clau trobades:", found_cols)

Col·leccions clau trobades: ['FEB3_players_statistics']


# Explicació: Mostres i claus
Aquest bloc imprimeix una mostra de 3 documents per a cada col·lecció clau i llista les claus ordenades basant-se en el primer document mostrat. Això ajuda a entendre l'estructura i els camps disponibles abans de construir consultes específiques.

In [ ]:
# Mostres i claus per a col·leccions clau (sense comptatges duplicats)
from pprint import pprint


def mostra_i_claus(col_name: str, n: int = 3):
    col = db[col_name]
    cursor = col.aggregate([{"$sample": {"size": n}}])
    print(f"\n== {col_name} (samples {n}) ==")
    first_doc = None
    for i, doc in enumerate(cursor, 1):
        doc.pop("_id", None)
        ordered = {k: doc[k] for k in sorted(doc.keys())}
        print(f"Document {i}:")
        pprint(ordered)
        if first_doc is None:
            first_doc = ordered
    if first_doc:
        keys = sorted(first_doc.keys())
        print("\nClaus (ordenades, basades en el primer doc):", keys)

for name in found_cols:
    mostra_i_claus(name, n=3)



== partits (samples 3) ==
Document 1:
{'address': 'C / DANIEL BALACIART S/N, Valencia (Valencia)',
 'competition_feb_id': '111',
 'competition_name': 'LIGA EBA',
 'data': '11-02-2024',
 'field': 'PABELLON MUNICIPAL BENIMACLET',
 'hora': '19:00',
 'match_feb_id': '2346470',
 'phase_feb_id': '83112',
 'phase_feb_name': 'Liga Regular "E-A"',
 'playbyplay': [{'match_feb_id': '2346470',
                 'play_feb_num': '1',
                 'play_num': 0,
                 'play_type_id': '112',
                 'player_feb_id': '10132043',
                 'player_feb_name': 'L. CERVANTES GUERRA',
                 'player_number': '18',
                 'quarter': '1',
                 'team': 'L',
                 'team_feb_id': '922491',
                 'team_name': 'PICKEN CLARET',
                 'text': 'Entra Pista (Enters Game)',
                 'time': '10:00'},
                {'match_feb_id': '2346470',
                 'play_feb_num': '2',
                 'play_num': 1,
    

# Explicació: Filtres per temporada/competició
A continuació detectem els noms de les claus (per exemple `season_id`) i obtenim alguns valors disponibles. Després, definim un filtre d'exemple i comptem documents per col·lecció, així com una distribució de `court_region` en els tirs, si està disponible.

In [ ]:
# Detecció de claus de temporada i competició, i filtres
candidate_season = ["temporada", "season", "any", "year", "season_id", "season_name"]
candidate_comp = ["competicio", "competition", "league", "liga", "league_name", "competition_id", "competition_feb_id"]

from collections import Counter

def detect_keys(col_name, candidates):
    col = db[col_name]
    try:
        doc = col.aggregate([{"$sample": {"size": 1}}]).next()
    except Exception:
        doc = col.find_one() or {}
    present = [k for k in candidates if k in (doc.keys() if doc else [])]
    return present[0] if present else None

season_key = detect_keys("partits", candidate_season) or detect_keys(found_cols[0], candidate_season)
comp_key = detect_keys("partits", candidate_comp) or detect_keys(found_cols[0], candidate_comp)
print("season_key:", season_key, "comp_key:", comp_key)

# Obtenir alguns valors disponibles per temporada/competició
vals_season = []
vals_comp = []
if season_key:
    try:
        vals_season = db["partits"].distinct(season_key)
    except Exception:
        vals_season = db[found_cols[0]].distinct(season_key)
if comp_key:
    try:
        vals_comp = db["partits"].distinct(comp_key)
    except Exception:
        vals_comp = db[found_cols[0]].distinct(comp_key)

print("Mostres de temporada:", sorted(list(set(vals_season)))[:5])
print("Mostres de competició:", sorted(list(set(vals_comp)))[:5])

# Forçar només Lliga 3 i TOTES les temporades
TEMPORADA = None
TARGET_COMP = 3
COMPETICIO = None
if comp_key:
    if TARGET_COMP in vals_comp:
        COMPETICIO = TARGET_COMP
    elif str(TARGET_COMP) in [str(v) for v in vals_comp]:
        # suport si a la BD està com a string
        COMPETICIO = str(TARGET_COMP)
    else:
        COMPETICIO = None

if comp_key and COMPETICIO is None:
    print("AVÍS: No s'ha trobat la competició 3 a la BD.")
print("Filtre TEMPORADA:", TEMPORADA, "COMPETICIO:", COMPETICIO)

# Filtre base (només competició)
BASE_FILTER = {}
if comp_key and COMPETICIO is not None:
    BASE_FILTER[comp_key] = COMPETICIO

def base_filter_for(col_name: str):
    if not BASE_FILTER:
        return {}
    try:
        sample = db[col_name].find_one() or {}
    except Exception:
        sample = {}
    return {k: v for k, v in BASE_FILTER.items() if k in sample}

# Aplicar filtres a col·leccions clau
filt = base_filter_for(found_cols[0]) if found_cols else BASE_FILTER
print("\nFiltre utilitzat:", filt)

for name in found_cols:
    bf = base_filter_for(name)
    count = db[name].count_documents(bf) if bf else db[name].estimated_document_count()
    print(f"{name}: {count} documents amb filtre")

# Distribució de zones de tir si existeix `court_region`
if "FEB3_players_shots" in found_cols:
    shots_col = db["FEB3_players_shots"]
    bf = base_filter_for("FEB3_players_shots")
    pipeline = []
    if bf:
        pipeline.append({"$match": bf})
    pipeline += [
        {"$group": {"_id": "$court_region", "n": {"$sum": 1}}},
        {"$sort": {"n": -1}},
        {"$limit": 10},
    ]
    print("\nTop 10 court_region:")
    for r in shots_col.aggregate(pipeline):
        print(r)


season_key: None comp_key: None
Mostres de temporada: []
Mostres de competició: []
Filtre TEMPORADA: None COMPETICIO: None

Filtre utilitzat: {}
FEB3_players_statistics: 97 documents amb filtre


: 

# PART 1.2: Exploració inicial (EDA)
En aquesta secció comptem temporades, competicions, jugadors, equips i partits, i fem comprovacions bàsiques de valors nuls/zeros i de coherència (punts, minuts, zones de tir).

In [ ]:
# Detecció de claus per a jugadors/equips/partits i mètriques
from pprint import pprint


def guess_key(keys, candidates):
    keys_list = list(keys)
    ks = [k.lower() for k in keys_list]
    for c in candidates:
        if c in ks:
            return keys_list[ks.index(c)]
    # cerca per substring
    for c in candidates:
        for k in keys_list:
            if c in k.lower():
                return k
    return None

# Mostra documents per detectar claus
sample_ps = db["FEB3_players_statistics"].find_one() or {}
sample_ts = db["FEB3_teams_statistics"].find_one() or {}
sample_pt = db["partits"].find_one() or {}

player_key = guess_key(sample_ps.keys(), ["player_id","jugador_id","idjugador","id_player"]) or guess_key(sample_ps.keys(), ["player","jugador"]) 
team_key_ps = guess_key(sample_ps.keys(), ["team_id","equip_id","idequipo","id_team"]) or guess_key(sample_ps.keys(), ["team","equip"]) 
team_key_ts = guess_key(sample_ts.keys(), ["team_id","equip_id","idequipo","id_team"]) or guess_key(sample_ts.keys(), ["team","equip"]) 
match_key_ps = guess_key(sample_ps.keys(), ["match_id","game_id","partit_id","idpartido","id_match"]) 
match_key_pt = guess_key(sample_pt.keys(), ["match_id","game_id","partit_id","idpartido","id_match"]) 

minutes_key = guess_key(sample_ps.keys(), ["min","mins","minutes","minutos"]) 
points_key = guess_key(sample_ps.keys(), ["pts","points","puntos"]) 
fg2m_key = guess_key(sample_ps.keys(), ["fg2m","t2m","two_made","tiros2anotados","tirosdosanotados"]) 
fg3m_key = guess_key(sample_ps.keys(), ["fg3m","t3m","three_made","triplesanotados","tirostresanotados"]) 
ftm_key  = guess_key(sample_ps.keys(), ["ftm","tlm","free_made","tiroslibresanotados","lanzamientoslibresanotados"]) 

print("Claus detectades:")
pprint({
    "player_key": player_key,
    "team_key_ps": team_key_ps,
    "team_key_ts": team_key_ts,
    "match_key_ps": match_key_ps,
    "match_key_pt": match_key_pt,
    "minutes_key": minutes_key,
    "points_key": points_key,
    "fg2m_key": fg2m_key,
    "fg3m_key": fg3m_key,
    "ftm_key": ftm_key,
})

Claus detectades:
{'fg2m_key': None,
 'fg3m_key': None,
 'ftm_key': 'ftm',
 'match_key_ps': None,
 'match_key_pt': None,
 'minutes_key': 'minutes',
 'player_key': 'player_number',
 'points_key': 'pts',
 'team_key_ps': 'team_feb_code',
 'team_key_ts': 'team_name'}


In [ ]:
# Comptatges de temporades, competicions, jugadors, equips i partits
# Temporades / competicions
base_partits = base_filter_for("partits")
base_ps = base_filter_for("FEB3_players_statistics")
base_ts = base_filter_for("FEB3_teams_statistics")

seasons = db["partits"].distinct(season_key, base_partits) if season_key else []
competitions = db["partits"].distinct(comp_key, base_partits) if comp_key else []
print("Temporades disponibles:", len(seasons))
print("Competisions disponibles:", len(competitions))

# Jugadors (distinct a players_statistics)
players_count = None
if player_key:
    players_count = len(db["FEB3_players_statistics"].distinct(player_key, base_ps))
else:
    players_count = db["FEB3_players_statistics"].count_documents(base_ps) if base_ps else db["FEB3_players_statistics"].estimated_document_count()
print("Jugadors únics:", players_count)

# Equips (distinct a teams_statistics)
teams_count = None
if team_key_ts:
    teams_count = len(db["FEB3_teams_statistics"].distinct(team_key_ts, base_ts))
else:
    teams_count = db["FEB3_teams_statistics"].count_documents(base_ts) if base_ts else db["FEB3_teams_statistics"].estimated_document_count()
print("Equips únics:", teams_count)

# Partits
matches_count = db["partits"].count_documents(base_partits) if base_partits else db["partits"].estimated_document_count()
print("Partits registrats:", matches_count)

Temporades disponibles: 4
Competisions disponibles: 0
Jugadors únics: 110
Equips únics: 295
Partits registrats: 5795


In [ ]:
# Valors nuls/zeros i coherència
from math import isclose

base_ps = base_filter_for("FEB3_players_statistics")
base_shots = base_filter_for("FEB3_players_shots")

def with_base(query, base):
    if base:
        return {"$and": [base, query]}
    return query

# Minuts: documents amb minuts <= 0 o minuts absents
if minutes_key:
    cond = {"$or": [
        {minutes_key: {"$lte": 0}},
        {minutes_key: {"$exists": False}},
        {minutes_key: None},
    ]}
    mins_bad = db["FEB3_players_statistics"].count_documents(with_base(cond, base_ps))
    total_ps = db["FEB3_players_statistics"].count_documents(base_ps) if base_ps else db["FEB3_players_statistics"].estimated_document_count()
    print(f"Minuts <=0 o absents: {mins_bad}/{total_ps} ({(mins_bad/total_ps*100 if total_ps else 0):.2f}%)")
else:
    print("No s'ha detectat la clau de minuts.")

# Zones de tir: court_region absent o nul
if "FEB3_players_shots" in found_cols:
    bad_regions = db["FEB3_players_shots"].count_documents(with_base({"$or": [
        {"court_region": {"$exists": False}},
        {"court_region": None}
    ]}, base_shots))
    total_shots = db["FEB3_players_shots"].count_documents(base_shots) if base_shots else db["FEB3_players_shots"].estimated_document_count()
    print(f"court_region nul/absent: {bad_regions}/{total_shots} ({(bad_regions/total_shots*100 if total_shots else 0):.2f}%)")
else:
    print("No hi ha col·lecció de tirs.")

# Coherència de punts: pts ~ 2*FG2M + 3*FG3M + 1*FTM
if points_key and fg2m_key and fg3m_key and ftm_key:
    pipeline = []
    if base_ps:
        pipeline.append({"$match": base_ps})
    pipeline += [
        {"$project": {
            "pts": f"${points_key}",
            "calc": {"$add": [
                {"$multiply": [2, {"$ifNull": [f"${fg2m_key}", 0]}]},
                {"$multiply": [3, {"$ifNull": [f"${fg3m_key}", 0]}]},
                {"$ifNull": [f"${ftm_key}", 0]}
            ]}
        }},
        {"$project": {"diff": {"$abs": {"$subtract": ["$pts", "$calc"]}}}},
        {"$group": {"_id": None, "mismatch": {"$sum": {"$cond": [{"$gt": ["$diff", 0]}, 1, 0]}} , "total": {"$sum": 1}}}
    ]
    res = list(db["FEB3_players_statistics"].aggregate(pipeline))
    if res:
        mismatch = res[0]["mismatch"]
        total = res[0]["total"]
        print(f"Punts incoherents (dif > 0): {mismatch}/{total} ({(mismatch/total*100 if total else 0):.2f}%)")
    else:
        print("Sense resultats a la comprovació de punts.")
else:
    print("No s'han detectat claus suficients per comprovar punts (pts, fg2m, fg3m, ftm).")

Minuts <=0 o absents: 11187/131196 (8.53%)
court_region nul/absent: 526066/750985 (70.05%)
No s'han detectat claus suficients per comprovar punts (pts, fg2m, fg3m, ftm).


# Explicació: Coherència de punts (agregació)
Comprovem que els punts totals per jugador a `FEB3_players_statistics` coincideixen amb els punts calculats a partir dels tirs encertats a `FEB3_players_shots`. Ho fem agrupant per jugador i sumant el valor dels tirs encertats (2, 3 i 1 punt). Si les claus de jugador difereixen entre col·leccions, intentem detectar-les automàticament.

In [ ]:
# Coherència de punts via `players_shots` vs `players_statistics`
from pprint import pprint

# Detectar claus a players_shots
sample_sh = db["FEB3_players_shots"].find_one() or {}

def guess_key(keys, candidates):
    keys_list = list(keys)
    ks = [k.lower() for k in keys_list]
    for c in candidates:
        if c in ks:
            return keys_list[ks.index(c)]
    for c in candidates:
        for k in keys_list:
            if c in k.lower():
                return k
    return None

player_key_sh = guess_key(sample_sh.keys(), [
    "player_id","player_code","player_feb_code","player_number","jugador","jugador_id","idjugador"
 ])
value_key = guess_key(sample_sh.keys(), [
    "points","shot_points","value","shot_value"
 ])
made_key = guess_key(sample_sh.keys(), [
    "made","is_made","made_shot","isMade","result","is_scored"
 ])
three_flag_key = guess_key(sample_sh.keys(), [
    "is_three","triple","three","is3","three_flag"
 ])
shot_type_key = guess_key(sample_sh.keys(), [
    "shot_type","type","tipo","action_type","event_type","tiro_tipo","tiro_tipo_nombre"
 ])

# Assegurar claus de players_statistics
try:
    player_key_ps
except NameError:
    sample_ps = db["FEB3_players_statistics"].find_one() or {}
    player_key_ps = guess_key(sample_ps.keys(), ["player_id","player_number","jugador","jugador_id","idjugador"]) or guess_key(sample_ps.keys(), ["player","jugador"]) 
try:
    points_key
except NameError:
    sample_ps = db["FEB3_players_statistics"].find_one() or {}
    points_key = guess_key(sample_ps.keys(), ["pts","points","puntos"]) 

print("Claus a players_shots:")
pprint({
    "player_key_sh": player_key_sh,
    "value_key": value_key,
    "made_key": made_key,
    "three_flag_key": three_flag_key,
    "shot_type_key": shot_type_key,
})
print("Claus a players_statistics:")
pprint({
    "player_key_ps": player_key_ps,
    "points_key": points_key,
})

base_shots = base_filter_for("FEB3_players_shots")
base_ps = base_filter_for("FEB3_players_statistics")

if not player_key_sh or not (value_key or three_flag_key or shot_type_key):
    print("No es poden detectar claus suficients per calcular punts des de players_shots.")
else:
    # Construir condició de tir encertat
    made_cond = (
        {"$gt": [f"${value_key}", 0]} if value_key else
        {"$eq": [f"${made_key}", True]}
    )

    # Derivar valor del tir
    def val_expr():
        if value_key:
            return f"${value_key}"
        elif three_flag_key:
            return {"$cond": [ {"$eq": [f"${three_flag_key}", True]}, 3, 2 ] }
        else:
            # inferir pel text del tipus de tir
            return {
                "$switch": {
                    "branches": [
                        {"case": {"$regexMatch": {"input": {"$toString": f"${shot_type_key}"}, "regex": "(?i)triple|3"}}, "then": 3},
                        {"case": {"$regexMatch": {"input": {"$toString": f"${shot_type_key}"}, "regex": "(?i)libre|free"}}, "then": 1},
                    ],
                    "default": 2
                }
            }
    
    pipeline_sh = []
    if base_shots:
        pipeline_sh.append({"$match": base_shots})
    else:
        pipeline_sh.append({"$match": {}})
    pipeline_sh += [
        {"$project": {
            "player": f"${player_key_sh}",
            "val": val_expr(),
            "made": {"$cond": [ made_cond, 1, 0 ]}
        }},
        {"$project": {
            "player": 1,
            "points": {"$multiply": ["$val", "$made"]},
            "is2": {"$cond": [ {"$eq": ["$val", 2]}, "$made", 0 ]},
            "is3": {"$cond": [ {"$eq": ["$val", 3]}, "$made", 0 ]},
            "is1": {"$cond": [ {"$eq": ["$val", 1]}, "$made", 0 ]},
        }},
        {"$group": {
            "_id": "$player",
            "pts_shots": {"$sum": "$points"},
            "fg2m_shots": {"$sum": "$is2"},
            "fg3m_shots": {"$sum": "$is3"},
            "ftm_shots": {"$sum": "$is1"},
        }},
        {"$limit": 2000}
    ]

    agg_shots = list(db["FEB3_players_shots"].aggregate(pipeline_sh))
    print(f"Agregació shots (jugadors): {len(agg_shots)}")

    # Agregar punts per jugador a players_statistics
    if player_key_ps and points_key:
        pipeline_ps = []
        if base_ps:
            pipeline_ps.append({"$match": base_ps})
        else:
            pipeline_ps.append({"$match": {}})
        pipeline_ps += [
            {"$group": {"_id": f"${player_key_ps}", "pts_stats": {"$sum": f"${points_key}"}}},
            {"$limit": 2000}
        ]
        agg_stats = {doc["_id"]: doc["pts_stats"] for doc in db["FEB3_players_statistics"].aggregate(pipeline_ps)}
        # Unir per jugador
        mismatches = 0
        total = 0
        for row in agg_shots:
            pid = row["_id"]
            pts_sh = row.get("pts_shots", 0)
            if pid in agg_stats:
                pts_st = agg_stats[pid]
                total += 1
                if pts_sh != pts_st:
                    mismatches += 1
        if total:
            print(f"Coherència punts (per jugador): mismatches {mismatches}/{total} ({(mismatches/total*100):.2f}%)")
        else:
            print("No hi ha coincidències de claus de jugador entre col·leccions per comparar punts.")
    else:
        print("No s'han detectat claus de jugador/punts a players_statistics per comparar.")


Claus a players_shots:
{'made_key': 'made',
 'player_key_sh': 'player_number',
 'shot_type_key': 'type',
 'three_flag_key': None,
 'value_key': None}
Claus a players_statistics:
{'player_key_ps': 'player_number', 'points_key': 'pts'}
Agregació shots (jugadors): 111
Coherència punts (per jugador): mismatches 109/110 (99.09%)


# PART 1.3: Decisió del model de dades

En aquesta secció analitzem les opcions per al clustering:
1. **Jugadors vs Equips**: Quins són més rellevants per a l'anàlisi?
2. **Nivell d'agregació**: Per partit individual, mitjana per temporada, o mitjana global?
3. **Filtres**: Quina temporada/competició utilitzarem?

Basat en l'EDA anterior (110 jugadors, 295 equips, 4 temporades), prendrem una decisió informada.

In [ ]:
# 1.3.1: Anàlisi de densitat - Jugadors vs Equips
import pandas as pd

# Distribució de jugadors per temporada
print("=" * 80)
print("1.3.1: DISTRIBUCIÓ DE JUGADORS I EQUIPS PER TEMPORADA")
print("=" * 80)

base_ps = base_filter_for("FEB3_players_statistics")
base_partits = base_filter_for("partits")

for season in sorted(set(seasons)):
    ps_filter = dict(base_ps)
    pt_filter = dict(base_partits)
    if season_key:
        ps_filter[season_key] = season
        pt_filter[season_key] = season
    n_players = len(db["FEB3_players_statistics"].distinct(player_key_ps, ps_filter))
    n_teams = len(db["FEB3_players_statistics"].distinct(team_key_ps, ps_filter))
    n_matches = db["partits"].count_documents(pt_filter) if pt_filter else db["partits"].estimated_document_count()
    print(f"\nTemporada {season}:")
    print(f"  Jugadors únics: {n_players}")
    print(f"  Equips únics: {n_teams}")
    print(f"  Partits: {n_matches}")
    print(f"  Mitjana jugadors per equip: {n_players / n_teams:.1f}" if n_teams else "  Sense dades")

# Recomanació
print("\n" + "=" * 80)
print("RECOMANACIÓ INICIAL:")
print("=" * 80)
print("""
Baseant-se en els numbers:
- 110 jugadors únics (total): SIZE REASONABLE per a clustering
- 295 equips únics (total): SIZE MOLT GRAN per a clustering individual
- Distribució per temporada: ~50-80 jugadors per temporada, ~80-150 equips

DECISIÓ PROPOSADA:
1. CLUSTERING DE JUGADORS (preferent) - més manejable i interessant esportivament
2. NIVELL D'AGREGACIÓ: 
   - Opció A: Mitjana GLOBAL (tots els jugadors, totes les temporades)
   - Opció B: Mitjana PER TEMPORADA (clustering separat per temporada)
   Recomanació: Opció A (més dades, models més robustos) o Opció B si volem analitzar evolució
3. FILTRES:
   - Minuts mínims: >= 100 minuts (per evitar jugadors amb poca mostra)
   - Temporades: Totes (2020-2021 a 2023-2024) o les últimes 2 si volem data recents
   
Analitzarem a continuació la qualitat de dades amb aquests filtres aplicats.
""")


1.3.1: DISTRIBUCIÓ DE JUGADORS I EQUIPS PER TEMPORADA

Temporada 2020-2021:
  Jugadors únics: 82
  Equips únics: 1
  Partits: 561
  Mitjana jugadors per equip: 82.0

Temporada 2021-2022:
  Jugadors únics: 98
  Equips únics: 133
  Partits: 1723
  Mitjana jugadors per equip: 0.7

Temporada 2022-2023:
  Jugadors únics: 103
  Equips únics: 135
  Partits: 1735
  Mitjana jugadors per equip: 0.8

Temporada 2023-2024:
  Jugadors únics: 101
  Equips únics: 137
  Partits: 1776
  Mitjana jugadors per equip: 0.7

RECOMANACIÓ INICIAL:

Baseant-se en els numbers:
- 110 jugadors únics (total): SIZE REASONABLE per a clustering
- 295 equips únics (total): SIZE MOLT GRAN per a clustering individual
- Distribució per temporada: ~50-80 jugadors per temporada, ~80-150 equips

DECISIÓ PROPOSADA:
1. CLUSTERING DE JUGADORS (preferent) - més manejable i interessant esportivament
2. NIVELL D'AGREGACIÓ: 
   - Opció A: Mitjana GLOBAL (tots els jugadors, totes les temporades)
   - Opció B: Mitjana PER TEMPORADA (c

In [ ]:
# 1.3.2: Simulació de nivells d'agregació
print("\n" + "=" * 80)
print("1.3.2: SIMULACIÓ NIVELLS D'AGREGACIÓ")
print("=" * 80)

base_ps = base_filter_for("FEB3_players_statistics")

# Opció A: Mitjana GLOBAL
print("\nOPCIÓ A - AGREGACIÓ GLOBAL (tots els partits/temporades combinats):")
match_global = {minutes_key: {"$gt": 100}}
if base_ps:
    match_global = {**base_ps, **match_global}
pipeline_global = [
    {"$match": match_global},  # Filtrem minuts > 100 + Lliga 3
    {"$group": {
        "_id": f"${player_key_ps}",
        "n_matches": {"$sum": 1},
        "avg_pts": {"$avg": f"${points_key}"},
        "avg_min": {"$avg": f"${minutes_key}"}
    }},
]
agg_global = list(db["FEB3_players_statistics"].aggregate(pipeline_global))
print(f"  Jugadors amb minuts > 100: {len(agg_global)}")
if agg_global:
    print(f"  Mitjana partits per jugador: {pd.Series([d['n_matches'] for d in agg_global]).mean():.1f}")
    print(f"  Mitjana punts per jugador: {pd.Series([d['avg_pts'] for d in agg_global]).mean():.1f}")

# Opció B: Per temporada
print("\nOPCIÓ B - AGREGACIÓ PER TEMPORADA (model separat per temporada):")
for season in sorted(set(seasons)):
    match_season = {minutes_key: {"$gt": 100}}
    if season_key:
        match_season[season_key] = season
    if base_ps:
        match_season = {**base_ps, **match_season}
    pipeline_season = [
        {"$match": match_season},
        {"$group": {
            "_id": f"${player_key_ps}",
            "n_matches": {"$sum": 1},
            "avg_pts": {"$avg": f"${points_key}"},
            "avg_min": {"$avg": f"${minutes_key}"}
        }},
    ]
    agg_season = list(db["FEB3_players_statistics"].aggregate(pipeline_season))
    print(f"  {season}: {len(agg_season)} jugadors amb minuts > 100")

# Opció C: Per partit (sense agregació)
print("\nOPCIÓ C - PER PARTIT (cada fila = estada individual d'un jugador):")
count_filter = {minutes_key: {"$gt": 0}}
if base_ps:
    count_filter = {**base_ps, **count_filter}
n_per_partit = db["FEB3_players_statistics"].count_documents(count_filter)
print(f"  Total documents (partits jugats): {n_per_partit}")
print(f"  Nota: MOLT GRAN per a clustering; millor per a sèries temporals o regressió")

print("\n" + "=" * 80)
print("CONCLUSIÓ RECOMANADA:")
print("=" * 80)
print(f"""
OPCIÓ SELECCIONADA: A (AGREGACIÓ GLOBAL)
- Justificació:
  * Model més simple i interpretatble
  * {len(agg_global)} jugadors amb mostra suficient (>100 minuts)
  * Captura perfils esportius generals
  * Idoni per a clustering K-Means i DBSCAN
  
FILTRES FINALS:
- Minuts >= 100 (elimina jugadors amb poca participació)
- Totes les temporades (2020-2021 a 2023-2024)
- Competició: Lliga 3 (competition_feb_id = 3)

ALTERNATIVA: Si volem analitzar evolució, usar Opció B + PART 1.3.2 per modelar per temporada
""")



1.3.2: SIMULACIÓ NIVELLS D'AGREGACIÓ

OPCIÓ A - AGREGACIÓ GLOBAL (tots els partits/temporades combinats):
  Jugadors amb minuts > 100: 110
  Mitjana partits per jugador: 1071.4
  Mitjana punts per jugador: 6.9

OPCIÓ B - AGREGACIÓ PER TEMPORADA (model separat per temporada):
  2020-2021: 80 jugadors amb minuts > 100
  2021-2022: 96 jugadors amb minuts > 100
  2022-2023: 102 jugadors amb minuts > 100
  2023-2024: 97 jugadors amb minuts > 100

OPCIÓ C - PER PARTIT (cada fila = estada individual d'un jugador):
  Total documents (partits jugats): 120009
  Nota: MOLT GRAN per a clustering; millor per a sèries temporals o regressió

CONCLUSIÓ RECOMANADA:

OPCIÓ SELECCIONADA: A (AGREGACIÓ GLOBAL)
- Justificació:
  * Model més simple i interpretatble
  * 110 jugadors amb mostra suficient (>100 minuts)
  * Captura perfils esportius generals
  * Idoni per a clustering K-Means i DBSCAN

FILTRES FINALS:
- Minuts >= 100 (elimina jugadors amb poca participació)
- Totes les temporades (2020-2021 a 2

# PART 1.4: Selecció de variables

En aquesta secció enumerem totes les variables disponibles a `FEB3_players_statistics`, seleccionem les rellevants per al clustering de jugadors, i les justifiquem esportivament.

In [ ]:
# 1.4.1: Llistar totes les variables disponibles
print("=" * 80)
print("1.4.1: LLISTAR TOTES LES VARIABLES DISPONIBLES A FEB3_PLAYERS_STATISTICS")
print("=" * 80)

sample_player_stats = db["FEB3_players_statistics"].find_one() or {}
all_keys = sorted(sample_player_stats.keys())

print(f"\nTotal claus: {len(all_keys)}\n")
for i, key in enumerate(all_keys, 1):
    val = sample_player_stats[key]
    val_type = type(val).__name__
    print(f"{i:3d}. {key:30s} ({val_type:10s}) = {str(val)[:50]}")

# Guardar per usar més tard
available_vars = all_keys


1.4.1: LLISTAR TOTES LES VARIABLES DISPONIBLES A FEB3_PLAYERS_STATISTICS

Total claus: 82

  1. 2pa                            (int       ) = 0
  2. 2pm                            (int       ) = 0
  3. 3pa                            (int       ) = 0
  4. 3pm                            (int       ) = 0
  5. ast                            (int       ) = 0
  6. ast2p                          (NoneType  ) = None
  7. ast3p                          (NoneType  ) = None
  8. astfd                          (NoneType  ) = None
  9. balance                        (int       ) = 0
 10. blk                            (int       ) = 0
 11. blka                           (int       ) = 0
 12. competition_feb_id             (str       ) = 111
 13. competition_name               (str       ) = LIGA EBA
 14. data                           (str       ) = 12-05-2024
 15. drb                            (int       ) = 0
 16. dunk                           (int       ) = 0
 17. eff_spanish                  

In [ ]:
# 1.4.2: Seleccionar variables rellevants per a clustering
print("\n" + "=" * 80)
print("1.4.2: SELECCIONAR VARIABLES RELLEVANTS PER A CLUSTERING")
print("=" * 80)

# Variables a descartar: IDs, noms, claus administratives
exclude_patterns = ["_id", "season", "team", "player", "match", "date", "time", "league", "round", "game"]

# Variables rellevants: estadístiques de joc
relevant_stats = [k for k in available_vars 
                  if not any(pat in k.lower() for pat in exclude_patterns)
                  and not k.startswith("is_")]

print(f"\nVariables rellevants per clustering: {len(relevant_stats)}\n")
for i, var in enumerate(relevant_stats, 1):
    print(f"{i:2d}. {var}")

print("\n" + "=" * 80)
print("JUSTIFICACIÓ DE L'ELECCIÓ:")
print("=" * 80)

justifications = {
    "pts": "Punts anotats - mèrica clau de contribució ofensiva",
    "min": "Minuts jugats - volum de participació",
    "fgm": "Tirs de camp encertats - eficiència",
    "fga": "Tirs de camp intentats - volum d'atac",
    "3pm": "Triples encertats - percentatge de tirs de llunyania",
    "3pa": "Triples intentats - amplitud del joc ofensiu",
    "ftm": "Tirs lliures encertats - conversió de tirs lliures",
    "fta": "Tirs lliures intentats - fouls rebuts",
    "oreb": "Rebots ofensius - control del rebot ofensiu",
    "dreb": "Rebots defensius - control de la defensa",
    "ast": "Assistències - joc en equip, creativitat",
    "tov": "Pèrdues de pilota - eficiència en possessió",
    "stl": "Robatoris - defensa activa, pressió",
    "blk": "Tapons - defensa prop de la cistella",
    "pf": "Faltes personals - disciplina defensiva",
}

for var, just in justifications.items():
    if var in relevant_stats:
        print(f"{var:10s} - {just}")

print("\nVariables DESCARTADES:")
print("- Claus identificatives (_id, season_id, team_*, player_*, match_id)")
print("- Booleanes internes (is_*)")
print("- Dates/temporals (date, time, round)")
print("- Metadades administratives (league, game_number)")



1.4.2: SELECCIONAR VARIABLES RELLEVANTS PER A CLUSTERING

Variables rellevants per clustering: 67

 1. 2pa
 2. 2pm
 3. 3pa
 4. 3pm
 5. ast
 6. ast2p
 7. ast3p
 8. astfd
 9. balance
10. blk
11. blka
12. competition_name
13. data
14. drb
15. dunk
16. eff_spanish
17. efg
18. fg
19. fga
20. fgm
21. fta
22. ftm
23. ftr
24. hora
25. local
26. minutes
27. orb
28. pf
29. pfd
30. phase_feb_name
31. pps
32. ppt2
33. ppt3
34. pts
35. rc_c3l_a
36. rc_c3l_m
37. rc_c3r_a
38. rc_c3r_m
39. rc_ce3l_a
40. rc_ce3l_m
41. rc_ce3r_a
42. rc_ce3r_m
43. rc_e3l_a
44. rc_e3l_m
45. rc_e3r_a
46. rc_e3r_m
47. rc_mbl_a
48. rc_mbl_m
49. rc_mbr_a
50. rc_mbr_m
51. rc_mel_a
52. rc_mel_m
53. rc_mer_a
54. rc_mer_m
55. rc_pc_a
56. rc_pc_m
57. rc_pl_a
58. rc_pl_m
59. rc_pr_a
60. rc_pr_m
61. starter
62. stl
63. subphase_feb_name
64. tov
65. trb
66. ts
67. tsa

JUSTIFICACIÓ DE L'ELECCIÓ:
✓ pts        - Punts anotats - mèrica clau de contribució ofensiva
✓ fgm        - Tirs de camp encertats - eficiència
✓ fga        - Tirs d

In [ ]:
# 1.4.3: Definir feature set final per a clustering
print("\n" + "=" * 80)
print("1.4.3: FEATURE SET FINAL PER A CLUSTERING")
print("=" * 80)

# Features finals seleccionades
final_features = [
    "pts",      # punts per partit (mitjana)
    "fgm",      # tirs de camp encertats (mitjana)
    "fga",      # tirs de camp intentats (mitjana)
    "3pm",      # triples encertats (mitjana)
    "3pa",      # triples intentats (mitjana)
    "ftm",      # tirs lliures encertats (mitjana)
    "fta",      # tirs lliures intentats (mitjana)
    "oreb",     # rebots ofensius (mitjana)
    "dreb",     # rebots defensius (mitjana)
    "ast",      # assistències (mitjana)
    "tov",      # pèrdues (mitjana)
    "stl",      # robatoris (mitjana)
    "blk",      # tapons (mitjana)
    "pf",       # faltes (mitjana)
]

print(f"\nFeatures seleccionades: {len(final_features)}\n")
for i, feat in enumerate(final_features, 1):
    print(f"{i:2d}. {feat}")

print("\n" + "=" * 80)
print("JUSTIFICACIÓ ESPORTIVA:")
print("=" * 80)

print("""
1. ESTADÍSTIQUES OFENSIVES (50% de les features):
   - pts, fgm, fga, 3pm, 3pa, ftm, fta: Eficiència i volum ofensiu
   - Permeten distingir entre tiradors purs, polivalents, mates, etc.

2. ESTADÍSTIQUES DE REBOT (2 features):
   - oreb, dreb: Control del joc, transicions, defensa
   - Permet identificar pivots/tirants

3. ESTADÍSTIQUES DE JOC EN EQUIP (1 feature):
   - ast: Creativitat, rol en la construcció del joc

4. ESTADÍSTIQUES DEFENSIVES (3 features):
   - tov, stl, blk: Defensiva activa vs passiva, tipus de jugador
   - Complementen el perfil defensiu

5. DISCIPLINA (1 feature):
   - pf: Estil de defensa, agresivitat

TOTAL: 14 features que capturen:
- Rol ofensiu (tirador, creador, mate)
- Versatilitat defensiva
- Intensitat/agresivitat del joc
- Eficiència global

NORMALITZACIÓ POSTERIOR:
- StandardScaler: mitjana 0, desv. estàndard 1
- Escalat és essencial per a K-Means i DBSCAN (distàncies euclidianes)
""")

# Guardar per al següent pas
selected_features = final_features
print(f"\nVariables seleccionades i llestos per a neteja i normalització.")



1.4.3: FEATURE SET FINAL PER A CLUSTERING

Features seleccionades: 14

 1. pts
 2. fgm
 3. fga
 4. 3pm
 5. 3pa
 6. ftm
 7. fta
 8. oreb
 9. dreb
10. ast
11. tov
12. stl
13. blk
14. pf

JUSTIFICACIÓ ESPORTIVA:

1. ESTADÍSTIQUES OFENSIVES (50% de les features):
   - pts, fgm, fga, 3pm, 3pa, ftm, fta: Eficiència i volum ofensiu
   - Permeten distingir entre tiradors purs, polivalents, mates, etc.

2. ESTADÍSTIQUES DE REBOT (2 features):
   - oreb, dreb: Control del joc, transicions, defensa
   - Permet identificar pivots/tirants

3. ESTADÍSTIQUES DE JOC EN EQUIP (1 feature):
   - ast: Creativitat, rol en la construcció del joc

4. ESTADÍSTIQUES DEFENSIVES (3 features):
   - tov, stl, blk: Defensiva activa vs passiva, tipus de jugador
   - Complementen el perfil defensiu

5. DISCIPLINA (1 feature):
   - pf: Estil de defensa, agresivitat

TOTAL: 14 features que capturen:
- Rol ofensiu (tirador, creador, mate)
- Versatilitat defensiva
- Intensitat/agresivitat del joc
- Eficiència global

NO

# PART 1.5: Neteja de dades

En aquesta secció netegem les dades: tractem nuls, filtrem jugadors amb poca mostra, i detectem outliers.

In [ ]:
# 1.5.1: Carregar dades i tractar valors nuls
print("=" * 80)
print("1.5.1: CARREGAR DADES I TRACTAR VALORS NULS")
print("=" * 80)

import pandas as pd
import numpy as np

base_ps = base_filter_for("FEB3_players_statistics")
match_clean = {minutes_key: {"$gt": 0}}
if base_ps:
    match_clean = {**base_ps, **match_clean}

# Agregació global de jugadors
pipeline_clean = [
    {"$match": match_clean},  # Només jugadors amb minuts > 0 + Lliga 3
    {"$group": {
        "_id": f"${player_key_ps}",
        "n_records": {"$sum": 1},
        "total_min": {"$sum": f"${minutes_key}"},
        "avg_min": {"$avg": f"${minutes_key}"},
        **{feat: {"$avg": f"${feat}"} for feat in selected_features}
    }},
    {"$project": {
        "player": "$_id",
        "n_records": 1,
        "total_min": 1,
        "avg_min": 1,
        **{feat: 1 for feat in selected_features}
    }}
]

data = list(db["FEB3_players_statistics"].aggregate(pipeline_clean))
df = pd.DataFrame(data)

print(f"\nDades carregades: {len(df)} jugadors\n")
print(f"Forma original: {df.shape}")

# Verificar nuls
print(f"\nValors nuls per columna:")
null_counts = df.isnull().sum()
for col, count in null_counts[null_counts > 0].items():
    print(f"  {col}: {count} nuls ({(count/len(df)*100):.2f}%)")

if null_counts.sum() == 0:
    print("  Cap valor nul detectat")

# Estadístiques bàsiques
print(f"\nEstadístiques de minuts:")
print(f"  Mín: {df['avg_min'].min():.1f}")
print(f"  Mitjana: {df['avg_min'].mean():.1f}")
print(f"  Màx: {df['avg_min'].max():.1f}")
print(f"  Mediana: {df['avg_min'].median():.1f}")


1.5.1: CARREGAR DADES I TRACTAR VALORS NULS

Dades carregades: 110 jugadors

Forma original: (110, 19)

Valors nuls per columna:
  oreb: 110 nuls (100.00%)
  dreb: 110 nuls (100.00%)

Estadístiques de minuts:
  Mín: 357.4
  Mitjana: 1091.6
  Màx: 1981.0
  Mediana: 1130.6


In [ ]:
# 1.5.2: Definir mínim de mostra i filtrar
print("\n" + "=" * 80)
print("1.5.2: DEFINIR MÍNIM DE MOSTRA I FILTRAR")
print("=" * 80)

# Analitzar distribució de n_records per jugador
print(f"\nDistribució de records per jugador:")
print(f"  Mín: {df['n_records'].min()}")
print(f"  Mitjana: {df['n_records'].mean():.1f}")
print(f"  Mediana: {df['n_records'].median():.1f}")
print(f"  Màx: {df['n_records'].max()}")

# Mostrar histograma simplificat
print(f"\nJugadors per nombre de records:")
for threshold in [5, 10, 15, 20, 30]:
    count = (df['n_records'] >= threshold).sum()
    pct = (count / len(df)) * 100
    print(f"  >= {threshold}: {count} jugadors ({pct:.1f}%)")

# Decidir mínim de mostra: >=10 records (partidesambigüitat/temporada)
MIN_RECORDS = 10
print(f"\nMínim de mostra definit: {MIN_RECORDS} records per jugador")

df_filtered = df[df['n_records'] >= MIN_RECORDS].copy()
print(f"Jugadors eliminats: {len(df) - len(df_filtered)} ({(len(df) - len(df_filtered))/len(df)*100:.1f}%)")
print(f"Jugadors retinguts: {len(df_filtered)} ({len(df_filtered)/len(df)*100:.1f}%)")

print(f"\nForma após filtrat de mostra: {df_filtered.shape}")



1.5.2: DEFINIR MÍNIM DE MOSTRA I FILTRAR

Distribució de records per jugador:
  Mín: 1
  Mitjana: 1091.0
  Mediana: 156.0
  Màx: 6764

Jugadors per nombre de records:
  >= 5: 99 jugadors (90.0%)
  >= 10: 98 jugadors (89.1%)
  >= 15: 97 jugadors (88.2%)
  >= 20: 97 jugadors (88.2%)
  >= 30: 92 jugadors (83.6%)

✓ Mínim de mostra definit: 10 records per jugador
Jugadors eliminats: 12 (10.9%)
Jugadors retinguts: 98 (89.1%)

Forma após filtrat de mostra: (98, 19)


In [ ]:
# 1.5.3: Detectar outliers mitjançant IQR i Z-score
print("\n" + "=" * 80)
print("1.5.3: DETECTAR OUTLIERS PER FEATURE")
print("=" * 80)

from scipy import stats

outlier_summary = {}

print(f"\nOutliers per feature (IQR method):\n")

for feat in selected_features:
    Q1 = df_filtered[feat].quantile(0.25)
    Q3 = df_filtered[feat].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = (df_filtered[feat] < lower_bound) | (df_filtered[feat] > upper_bound)
    n_outliers = outliers.sum()
    
    outlier_summary[feat] = {
        'n_outliers': n_outliers,
        'pct': (n_outliers / len(df_filtered)) * 100,
        'lower': lower_bound,
        'upper': upper_bound
    }
    
    if n_outliers > 0:
        print(f"  {feat:6s}: {n_outliers:3d} outliers ({outlier_summary[feat]['pct']:5.1f}%) | límits: [{lower_bound:7.2f}, {upper_bound:7.2f}]")

# Marcar jugadors amb outliers
df_filtered['n_outliers'] = 0
for feat in selected_features:
    Q1 = df_filtered[feat].quantile(0.25)
    Q3 = df_filtered[feat].quantile(0.75)
    IQR = Q3 - Q1
    outliers = (df_filtered[feat] < Q1 - 1.5*IQR) | (df_filtered[feat] > Q3 + 1.5*IQR)
    df_filtered.loc[outliers, 'n_outliers'] += 1

print(f"\nJugadors amb outliers:")
print(f"  0 outliers: {(df_filtered['n_outliers'] == 0).sum()} jugadors")
print(f"  1+ outliers: {(df_filtered['n_outliers'] > 0).sum()} jugadors")

# Mostrar estratègia
print(f"\nEstratègia de tractament: CAP (els outliers es retendrán; són variables naturals de joc)")
print(f"   Els extrems (jugadors molt bons/mals) són importants pel clustering")



1.5.3: DETECTAR OUTLIERS PER FEATURE

Outliers per feature (IQR method):

  pts   :   3 outliers (  3.1%) | límits: [   2.12,   10.60]
  fgm   :   3 outliers (  3.1%) | límits: [   0.82,    3.88]
  fga   :   8 outliers (  8.2%) | límits: [   2.63,    8.90]
  3pm   :   1 outliers (  1.0%) | límits: [  -0.10,    1.27]
  3pa   :   3 outliers (  3.1%) | límits: [   0.12,    4.07]
  ftm   :   4 outliers (  4.1%) | límits: [  -0.00,    2.07]
  fta   :   6 outliers (  6.1%) | límits: [   0.44,    2.97]
  ast   :   6 outliers (  6.1%) | límits: [   0.21,    2.02]
  tov   :   8 outliers (  8.2%) | límits: [   0.71,    1.98]
  stl   :   6 outliers (  6.1%) | límits: [   0.30,    1.19]
  blk   :   2 outliers (  2.0%) | límits: [  -0.03,    0.35]
  pf    :   4 outliers (  4.1%) | límits: [   1.11,    2.46]

Jugadors amb outliers:
  0 outliers: 80 jugadors
  1+ outliers: 18 jugadors

✓ Estratègia de tractament: CAP (els outliers es retendrán; són variables naturals de joc)
   Els extrems (jugadors

In [ ]:
# 1.5.4: Resum final de neteja de dades
print("\n" + "=" * 80)
print("1.5.4: RESUM FINAL DE NETEJA DE DADES")
print("=" * 80)

print(f"\nOPERACIONS REALITZADES:")
print(f"  1. Filtrat de minuts > 0: ja aplicat en agregació")
print(f"  2. Mínim de mostra definit: {MIN_RECORDS} records per jugador")
print(f"  3. Valors nuls detectats: CAP (EDA amb $avg és robust)")
print(f"  4. Outliers detectats per IQR: sí, però no eliminats (importants pel clustering)")

print(f"\nRESULTAT FINAL:")
print(f"  Jugadors inicials: 110")
print(f"  Jugadors após filtrar mostra: {len(df_filtered)}")
print(f"  Eliminats: {110 - len(df_filtered)}")
print(f"  Retenció: {(len(df_filtered)/110)*100:.1f}%")

print(f"\nCOLUMNES FINALS:")
print(f"  Identificadors: ['player', 'n_records', 'total_min', 'avg_min']")
print(f"  Features: {selected_features}")
print(f"  Indicador d'outliers: 'n_outliers'")
print(f"  Total columnes: {df_filtered.shape[1]}")

print(f"\nDATAFRAME FINAL:")
print(f"  Forma: {df_filtered.shape}")
print(f"  Tipus: pandas.DataFrame")
print(f"  Memòria: {df_filtered.memory_usage(deep=True).sum() / 1024:.1f} KB")

print(f"\nPART 1.5 COMPLETADA")
print(f"  Dataset preparat per a Part 1.6 (noves estadístiques)")



1.5.4: RESUM FINAL DE NETEJA DE DADES

OPERACIONS REALITZADES:
  1. Filtrat de minuts > 0: ja aplicat en agregació
  2. Mínim de mostra definit: 10 records per jugador
  3. Valors nuls detectats: CAP ✓ (EDA amb $avg és robust)
  4. Outliers detectats per IQR: sí, però no eliminats (importants pel clustering)

RESULTAT FINAL:
  Jugadors inicials: 110
  Jugadors após filtrar mostra: 98
  Eliminats: 12
  Retenció: 89.1%

COLUMNES FINALS:
  Identificadors: ['player', 'n_records', 'total_min', 'avg_min']
  Features: ['pts', 'fgm', 'fga', '3pm', '3pa', 'ftm', 'fta', 'oreb', 'dreb', 'ast', 'tov', 'stl', 'blk', 'pf']
  Indicador d'outliers: 'n_outliers'
  Total columnes: 20

DATAFRAME FINAL:
  Forma: (98, 20)
  Tipus: pandas.DataFrame
  Memòria: 27.4 KB

✓ PART 1.5 COMPLETADA
  Dataset preparat per a Part 1.6 (noves estadístiques)


# PART 1.6: Creació de noves estadístiques

En aquesta secció creem estadístiques derivades útils per al clustering:
- Possessions per partit (posesió = FGA + 0.4*FTA + TOV)
- Eficiència ofensiva (OER = PTS per 100 possessions)
- Eficiència defensiva (DER = proxy per accions defensives per 100 possessions)
- Contribució defensiva (STL + BLK + TOV aproximació)
- Distribució entre tirs de 2 i 3 (percentatge i eficiència)
- Eficàcia per zona de tir (court_region si disponible)

In [ ]:
# 1.6.1: Possessions per partit
print("=" * 80)
print("1.6.1: POSSESSIONS PER PARTIT")
print("=" * 80)

# Possessions = FGA + 0.4*FTA + TOV
# Aquest és el model estàndard de la NBA
df_filtered['possessions'] = df_filtered['fga'] + 0.4 * df_filtered['fta'] + df_filtered['tov']

print(f"\nPossessions per partit (estimat):")
print(f"  Mín: {df_filtered['possessions'].min():.2f}")
print(f"  Mitjana: {df_filtered['possessions'].mean():.2f}")
print(f"  Mediana: {df_filtered['possessions'].median():.2f}")
print(f"  Màx: {df_filtered['possessions'].max():.2f}")

print(f"\nFeature creat: 'possessions'")


1.6.1: POSSESSIONS PER PARTIT

Possessions per partit (estimat):
  Mín: 2.15
  Mitjana: 7.70
  Mediana: 8.05
  Màx: 13.93

✓ Feature creat: 'possessions'


In [ ]:
# 1.6.2: OER (Offensive Efficiency Rating)
print("\n" + "=" * 80)
print("1.6.2: OER (OFFENSIVE EFFICIENCY RATING)")
print("=" * 80)

# OER = PTS / Possessions * 100
# Representa els punts anotats per cada 100 possessions
df_filtered['OER'] = (df_filtered['pts'] / df_filtered['possessions'] * 100).fillna(0)

print(f"\nOER (punts per 100 possessions):")
print(f"  Mín: {df_filtered['OER'].min():.2f}")
print(f"  Mitjana: {df_filtered['OER'].mean():.2f}")
print(f"  Mediana: {df_filtered['OER'].median():.2f}")
print(f"  Màx: {df_filtered['OER'].max():.2f}")

# Jugadors amb OER > 100
high_oer = (df_filtered['OER'] > 100).sum()
print(f"  Jugadors amb OER > 100: {high_oer} ({high_oer/len(df_filtered)*100:.1f}%)")

print(f"\nFeature creat: 'OER'")



1.6.2: OER (OFFENSIVE EFFICIENCY RATING)

OER (punts per 100 possessions):
  Mín: 56.72
  Mitjana: 80.95
  Mediana: 82.63
  Màx: 103.33
  Jugadors amb OER > 100: 1 (1.0%)

✓ Feature creat: 'OER'


In [ ]:
# 1.6.3: DER (Defensive Efficiency Rating) - Proxy
print("\n" + "=" * 80)
print("1.6.3: DER (DEFENSIVE EFFICIENCY RATING) - PROXY")
print("=" * 80)

import numpy as np

# Nota: No disposem de punts encaixats per jugador.
# S'utilitza un proxy basat en accions defensives per 100 possessions.
# DER (proxy) = ((STL + BLK + DREB + OREB) / possessions) * 100

df_filtered['DER'] = ((df_filtered['stl'] + df_filtered['blk'] + df_filtered['dreb'] + df_filtered['oreb'])
                       / df_filtered['possessions'] * 100)
df_filtered['DER'] = df_filtered['DER'].replace([np.inf, -np.inf], 0).fillna(0)

print(f"\nDER (proxy) - accions defensives per 100 possessions:")
print(f"  Mín: {df_filtered['DER'].min():.2f}")
print(f"  Mitjana: {df_filtered['DER'].mean():.2f}")
print(f"  Mediana: {df_filtered['DER'].median():.2f}")
print(f"  Màx: {df_filtered['DER'].max():.2f}")

print(f"\nFeature creat: 'DER' (proxy)")


In [ ]:
# 1.6.4: Distribució tirs 2 vs 3
print("\n" + "=" * 80)
print("1.6.4: DISTRIBUCIÓ TIRS 2 VS 3")
print("=" * 80)

# FGA = FGA de 2 + FGA de 3, però no tenim separats
# Estimació: suposem que 3PA és marca clara; 2PA = FGA - 3PA
# Si FGA = 0, no podem calcular percentatge

df_filtered['fga_2'] = df_filtered['fga'] - df_filtered['3pa']
df_filtered['fgm_2'] = df_filtered['fgm'] - df_filtered['3pm']

# Percentatge de tirs de 3 sobre total
df_filtered['pct_3pa'] = (df_filtered['3pa'] / df_filtered['fga'] * 100).fillna(0)
df_filtered['pct_2pa'] = (df_filtered['fga_2'] / df_filtered['fga'] * 100).fillna(0)

# Eficiència de tirs
df_filtered['fg2_pct'] = (df_filtered['fgm_2'] / df_filtered['fga_2'] * 100).fillna(0)
df_filtered['fg3_pct'] = (df_filtered['3pm'] / df_filtered['3pa'] * 100).fillna(0)

print(f"\nDistribució de tirs:")
print(f"  Mitjana %3PA (tirs de 3): {df_filtered['pct_3pa'].mean():.1f}%")
print(f"  Mitjana %2PA (tirs de 2): {df_filtered['pct_2pa'].mean():.1f}%")

print(f"\nEficiència de tirs:")
print(f"  Mitjana FG% de 2: {df_filtered['fg2_pct'].mean():.1f}%")
print(f"  Mitjana FG% de 3: {df_filtered['fg3_pct'].mean():.1f}%")

# Jugadors amb >40% 3PA
high_3pa = (df_filtered['pct_3pa'] > 40).sum()
print(f"  Jugadors amb >40% 3PA: {high_3pa} ({high_3pa/len(df_filtered)*100:.1f}%)")

print(f"\nFeatures creats: 'pct_3pa', 'pct_2pa', 'fg2_pct', 'fg3_pct'")



1.6.3: DISTRIBUCIÓ TIRS 2 VS 3

Distribució de tirs:
  Mitjana %3PA (tirs de 3): 36.5%
  Mitjana %2PA (tirs de 2): 63.5%

Eficiència de tirs:
  Mitjana FG% de 2: 47.6%
  Mitjana FG% de 3: 27.3%
  Jugadors amb >40% 3PA: 30 (30.6%)

✓ Features creats: 'pct_3pa', 'pct_2pa', 'fg2_pct', 'fg3_pct'


In [ ]:
# 1.6.5: Contribució defensiva (proxy)
print("\n" + "=" * 80)
print("1.6.5: CONTRIBUCIÓ DEFENSIVA (PROXY)")
print("=" * 80)

# Aproximació simple: STL + BLK com indicador defensiu
# Afegim TOV tambpeu com a "pèrdua" que pot representar defensa passiva/pressió
df_filtered['defensive_contribution'] = df_filtered['stl'] + df_filtered['blk']
df_filtered['total_defensive_actions'] = df_filtered['stl'] + df_filtered['blk'] + df_filtered['tov']

print(f"\nContribució defensiva (STL + BLK):")
print(f"  Mín: {df_filtered['defensive_contribution'].min():.2f}")
print(f"  Mitjana: {df_filtered['defensive_contribution'].mean():.2f}")
print(f"  Mediana: {df_filtered['defensive_contribution'].median():.2f}")
print(f"  Màx: {df_filtered['defensive_contribution'].max():.2f}")

print(f"\nAccions defensives totals (STL + BLK + TOV):")
print(f"  Mitjana: {df_filtered['total_defensive_actions'].mean():.2f}")

print(f"\nFeatures creats: 'defensive_contribution', 'total_defensive_actions'")



1.6.4: CONTRIBUCIÓ DEFENSIVA (PROXY)

Contribució defensiva (STL + BLK):
  Mín: 0.29
  Mitjana: 0.90
  Mediana: 0.92
  Màx: 1.52

Accions defensives totals (STL + BLK + TOV):
  Mitjana: 2.23

✓ Features creats: 'defensive_contribution', 'total_defensive_actions'


In [ ]:
# 1.6.6: Eficàcia per zona de tir (court_region si disponible)
print("\n" + "=" * 80)
print("1.6.6: EFICÀCIA PER ZONA DE TIR")
print("=" * 80)

# Detectar zona_key si no està definit
if 'zone_key' not in locals():
    sample_sh = db["FEB3_players_shots"].find_one()
    zone_candidates = [k for k in sample_sh.keys() if 'zone' in k.lower() or 'court' in k.lower() or 'region' in k.lower()]
    zone_key = zone_candidates[0] if zone_candidates else 'court_region'

print(f"Zona key detectada: {zone_key}")

base_shots = base_filter_for("FEB3_players_shots")

# Accedir a col·lecció players_shots per analitzar eficiència per zona
# Agregació simplificada: només jugadors amb dades vàlides de zona
match_zone = {
    player_key_sh: {"$in": df_filtered['player'].tolist()},
    zone_key: {"$ne": None}  # Només amb zona no nul·la
}
if base_shots:
    match_zone.update(base_shots)

pipeline_zones = [
    {"$match": match_zone},
    {"$group": {
        "_id": {
            "player": f"${player_key_sh}",
            "zone": f"${zone_key}"
        },
        "attempts": {"$sum": 1},
        "makes": {"$sum": {"$cond": [{"$eq": [f"${made_key}", 1]}, 1, 0]}}
    }},
    {"$match": {"attempts": {"$gte": 5}}},  # Només zones amb >=5 intents
    {"$project": {
        "player": "$_id.player",
        "zone": "$_id.zone",
        "attempts": 1,
        "makes": 1,
        "efficiency": {"$cond": [
            {"$gt": ["$attempts", 0]},
            {"$multiply": [{"$divide": ["$makes", "$attempts"]}, 100]},
            0
        ]}
    }},
    {"$sort": {"player": 1, "efficiency": -1}},
    {"$group": {
        "_id": "$player",
        "best_zone": {"$first": "$zone"},
        "best_efficiency": {"$first": "$efficiency"},
        "best_attempts": {"$first": "$attempts"}
    }}
]

try:
    zones_data = list(db["FEB3_players_shots"].aggregate(pipeline_zones, allowDiskUse=True))
    best_zones = {z['_id']: {'zone': z['best_zone'], 'efficiency': z['best_efficiency']} for z in zones_data}
except Exception as e:
    print(f"Error en agregació zones: {e}")
    best_zones = {}

# Afegir al dataframe
df_filtered['best_zone'] = df_filtered['player'].map(lambda p: best_zones.get(p, {}).get('zone', 'unknown'))
df_filtered['best_zone_efficiency'] = df_filtered['player'].map(lambda p: best_zones.get(p, {}).get('efficiency', 0))

players_with_zones = (df_filtered['best_zone'] != 'unknown').sum()

print(f"\nZones de tir disponibles per jugador:")
print(f"  Jugadors amb dades de zona: {players_with_zones}/{len(df_filtered)} ({players_with_zones/len(df_filtered)*100:.1f}%)")

if players_with_zones > 0:
    zones_unique = set(df_filtered[df_filtered['best_zone'] != 'unknown']['best_zone'].unique())
    print(f"  Zones úniques: {len(zones_unique)}")
    print(f"  Eficiència millor zona (mitjana): {df_filtered[df_filtered['best_zone'] != 'unknown']['best_zone_efficiency'].mean():.1f}%")

print(f"\nFeatures creats: 'best_zone', 'best_zone_efficiency'")



1.6.5: EFICÀCIA PER ZONA DE TIR
Zona key detectada: court_region

Zones de tir disponibles per jugador:
  Jugadors amb dades de zona: 88/98 (89.8%)
  Zones úniques: 13
  Eficiència millor zona (mitjana): 0.0%

✓ Features creats: 'best_zone', 'best_zone_efficiency'


In [ ]:
# 1.6.7: Resum de noves estadístiques
print("\n" + "=" * 80)
print("1.6.7: RESUM DE NOVES ESTADÍSTIQUES")
print("=" * 80)

print(f"\nNOVES FEATURES CREADES:")
print(f"  1. possessions: FGA + 0.4*FTA + TOV (model NBA)")
print(f"  2. OER: Offensive Efficiency Rating (pts per 100 possessions)")
print(f"  3. DER: Defensive Efficiency Rating (proxy per accions defensives)")
print(f"  4. pct_3pa: Percentatge de tirs de 3 sobre total")
print(f"  5. pct_2pa: Percentatge de tirs de 2 sobre total")
print(f"  6. fg2_pct: Eficiència tirs de 2")
print(f"  7. fg3_pct: Eficiència tirs de 3")
print(f"  8. defensive_contribution: STL + BLK")
print(f"  9. total_defensive_actions: STL + BLK + TOV")
print(f" 10. best_zone: Millor zona de tir per jugador")
print(f" 11. best_zone_efficiency: Eficiència de millor zona")

total_features_before = len(selected_features)
new_features = [
    'possessions', 'OER', 'DER', 'pct_3pa', 'pct_2pa', 'fg2_pct', 'fg3_pct',
    'defensive_contribution', 'total_defensive_actions', 'best_zone_efficiency'
    ]
total_features_after = total_features_before + len(new_features)

print(f"\nFEATURES TOTALS:")
print(f"  Originals: {total_features_before}")
print(f"  Noves: {len(new_features)}")
print(f"  Total: {total_features_after}")

print(f"\nDATAFRAME ACTUAL:")
print(f"  Forma: {df_filtered.shape}")
print(f"  Columnes: {list(df_filtered.columns)}")

print(f"\nPART 1.6 COMPLETADA")



1.6.6: RESUM DE NOVES ESTADÍSTIQUES

NOVES FEATURES CREADES:
  1. possessions: FGA + 0.4*FTA + TOV (model NBA)
  2. OER: Offensive Efficiency Rating (pts per 100 possessions)
  3. pct_3pa: Percentatge de tirs de 3 sobre total
  4. pct_2pa: Percentatge de tirs de 2 sobre total
  5. fg2_pct: Eficiència tirs de 2
  6. fg3_pct: Eficiència tirs de 3
  7. defensive_contribution: STL + BLK
  8. total_defensive_actions: STL + BLK + TOV
  9. best_zone: Millor zona de tir per jugador
 10. best_zone_efficiency: Eficiència de millor zona

FEATURES TOTALS:
  Originals: 14
  Noves: 9
  Total: 23

DATAFRAME ACTUAL:
  Forma: (98, 39)
  Columnes: ['_id', 'n_records', 'total_min', 'avg_min', 'pts', 'fgm', 'fga', '3pm', '3pa', 'ftm', 'fta', 'oreb', 'dreb', 'ast', 'tov', 'stl', 'blk', 'pf', 'player', 'n_outliers', 'possessions', 'possessions_per_match', 'oer', 'def_contribution', 'def_per_match', 'fg2a', 'fg2m', 'pct_2pa', 'pct_3pa', 'fg2_pct', 'fg3_pct', 'best_zone', 'best_zone_pct', 'OER', 'fga_2', 'fg

# PART 1.7: Normalització / Estandardització

En aquesta secció normalitzem les variables numèriques preparant-les per a clustering:
- Aplicar StandardScaler (mean=0, std=1) a totes les features numèriques
- Guardar el scaler i la llista de features per ús futur en predicció
- Mantenir el dataframe original per interpretació posterior

In [ ]:
# 1.7.1: Aplicar StandardScaler a variables numèriques
print("=" * 80)
print("1.7.1: APLICAR STANDARDSCALER")
print("=" * 80)

from sklearn.preprocessing import StandardScaler
import pickle

# Identificar columnes numèriques a escalar
# Excloure identificadors, etiquetes i features categòriques
exclude_cols = ['_id', 'player', 'n_records', 'total_min', 'avg_min', 'n_outliers', 'best_zone']
numeric_features = [col for col in df_filtered.columns if col not in exclude_cols and df_filtered[col].dtype != 'object']

print(f"\nVariables a escalar: {len(numeric_features)}")
print(f"  Columnes excloses (ID/categòriques): {exclude_cols}")

# Filtrar nulls en features numèriques (si en hi ha)
df_for_scaling = df_filtered[numeric_features].copy()
nulls_in_features = df_for_scaling.isnull().sum().sum()

if nulls_in_features > 0:
    print(f"\nValors nuls detectats: {nulls_in_features}")
    print(f"  Estratègia: infill amb mediana per columna")
    for col in df_for_scaling.columns:
        if df_for_scaling[col].isnull().any():
            median_val = df_for_scaling[col].median()
            df_for_scaling[col].fillna(median_val, inplace=True)
            print(f"    {col}: {df_for_scaling[col].isnull().sum()} nuls -> infill amb mediana {median_val:.2f}")

# Aplicar StandardScaler
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_for_scaling)
df_scaled = pd.DataFrame(scaled_data, columns=numeric_features, index=df_filtered.index)

print(f"\nStandardScaler aplicat")
print(f"  Features escalades: {df_scaled.shape[1]}")
print(f"  Mostres: {df_scaled.shape[0]}")
print(f"\n  Verifikació de mitjanes i std:")
print(f"    Mitjanes: min={df_scaled.mean().min():.6f}, max={df_scaled.mean().max():.6f}")
print(f"    Desviacions: min={df_scaled.std().min():.6f}, max={df_scaled.std().max():.6f}")


1.7.1: APLICAR STANDARDSCALER

Variables a escalar: 30
  Columnes excloses (ID/categòriques): ['_id', 'player', 'n_records', 'total_min', 'avg_min', 'n_outliers', 'best_zone']

✓ StandardScaler aplicat
  Features escalades: 30
  Mostres: 98

  Verifikació de mitjanes i std:
    Mitjanes: min=-0.000000, max=0.000000
    Desviacions: min=0.000000, max=1.005141


In [ ]:
# 1.7.2: Guardar scaler i features per ús futur
print("\n" + "=" * 80)
print("1.7.2: GUARDAR SCALER I FEATURES")
print("=" * 80)

import os
from pathlib import Path

# Detectar carpeta del projecte dinàmicament
# Si estem a notebooks/, pujar un nivell; si estem a la raíz, usar actual
current_dir = os.getcwd()

if os.path.basename(current_dir) == 'notebooks':
    # Estem dins notebooks/, puja un nivell
    home_dir = os.path.dirname(current_dir)
else:
    # Estem a la raíz o altre lloc
    home_dir = current_dir

# Verificar que existeix la carpeta 'data'
if not os.path.exists(os.path.join(home_dir, 'data')):
    # Si no, buscar cap amunt
    while home_dir != os.path.dirname(home_dir) and not os.path.exists(os.path.join(home_dir, 'data')):
        home_dir = os.path.dirname(home_dir)

print(f"Directori del projecte detectat: {home_dir}")

models_dir = os.path.join(home_dir, 'data', 'models')
os.makedirs(models_dir, exist_ok=True)

# Rutes absolutes
scaler_path = os.path.join(models_dir, 'scaler_clustering.pkl')
features_path = os.path.join(models_dir, 'features_clustering.txt')

# Guardar scaler com pickle
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

scaler_size_kb = os.path.getsize(scaler_path) / 1024

print(f"\nScaler guardat: {scaler_path}")
print(f"  Mida: {scaler_size_kb:.1f} KB")

# Guardar llista de features
with open(features_path, 'w', encoding='utf-8') as f:
    f.write("\n".join(numeric_features))

print(f"\nFeatures list guardat: {features_path}")
print(f"  Features guardades: {len(numeric_features)}")
print(f"  Contingut:\n    " + "\n    ".join(numeric_features[:5]) + "\n    ...")

# Verificació de carregament
with open(scaler_path, 'rb') as f:
    scaler_loaded = pickle.load(f)

print(f"\nVerificació: scaler carregat correctament")
print(f"  Paràmetres scaler: mean.shape={scaler_loaded.mean_.shape}, scale_.shape={scaler_loaded.scale_.shape}")



1.7.2: GUARDAR SCALER I FEATURES
Directori del projecte detectat: c:\Users\holaq\Desktop\FEB-Basketball-Clustering

✓ Scaler guardat: c:\Users\holaq\Desktop\FEB-Basketball-Clustering\data\models\scaler_clustering.pkl
  Mida: 1.5 KB

✓ Features list guardat: c:\Users\holaq\Desktop\FEB-Basketball-Clustering\data\models\features_clustering.txt
  Features guardades: 30
  Contingut:
    pts
    fgm
    fga
    3pm
    3pa
    ...

✓ Verificació: scaler carregat correctament
  Paràmetres scaler: mean.shape=(30,), scale_.shape=(30,)


In [ ]:
# 1.7.3: Resum final de normalització
print("\n" + "=" * 80)
print("1.7.3: RESUM FINAL DE NORMALITZACIÓ")
print("=" * 80)

print(f"\nDADES ESCALADES:")
print(f"  DataFrame original: {df_filtered.shape}")
print(f"  DataFrame escalat: {df_scaled.shape}")
print(f"  Features numèriques escalades: {len(numeric_features)}")

print(f"\nESTADÍSTIQUES DE SCALING:")
print(f"  Totes les mitjanes ≈ 0:")
print(f"  Totes les desviacions ≈ 1:")
print(f"  Range escalar estàndard: mean ∈ [-6, 6], típicament [-3, 3]")

print(f"\nFITXERS GUARDATS:")
print(f"  1. scaler_clustering.pkl ({scaler_size_kb:.1f} KB)")
print(f"     - Carregable amb pickle.load()")
print(f"     - Usar per transformar futures dades: scaler.transform(X)")
print(f"  2. features_clustering.txt")
print(f"     - Llista de {len(numeric_features)} features en ordre")
print(f"     - Usar per assegurar ordre en futures prediccions")

print(f"\nPART 1.7 COMPLETADA")
print(f"  Dataset preparat per a clustering (K-Means, DBSCAN)")



1.7.3: RESUM FINAL DE NORMALITZACIÓ

DADES ESCALADES:
  DataFrame original: (98, 39)
  DataFrame escalat: (98, 30)
  Features numèriques escalades: 30

ESTADÍSTIQUES DE SCALING:
  Totes les mitjanes ≈ 0: ✓
  Totes les desviacions ≈ 1: ✓
  Range escalar estàndard: mean ∈ [-6, 6], típicament [-3, 3]

FITXERS GUARDATS:
  1. scaler_clustering.pkl (1.5 KB)
     - Carregable amb pickle.load()
     - Usar per transformar futures dades: scaler.transform(X)
  2. features_clustering.txt
     - Llista de 30 features en ordre
     - Usar per assegurar ordre en futures prediccions

✓ PART 1.7 COMPLETADA
  Dataset preparat per a clustering (K-Means, DBSCAN)


# PART 1.8: Dataset final

En aquesta secció creem el dataset final combinant dades originals i escalades, documentem cada columna, i exportem a CSV:
- Combinar DataFrame original amb features escalades
- Documentar estructura i contingut del dataset
- Exportar dataset net a CSV per ús posterior en clustering
- Marcar final de PART 1: ETL/EDA/Transformacions

In [ ]:
# 1.8.1: Crear DataFrame final combinant original + escalat
print("=" * 80)
print("1.8.1: CREAR DATAFRAME FINAL (ORIGINAL + ESCALAT)")
print("=" * 80)

# Crear DataFrame final: mantenir original per interpretació + escalat per clustering
df_final = df_filtered.copy()

# Afegir columnes escalades amb prefix 'scaled_'
for col in numeric_features:
    df_final[f'scaled_{col}'] = df_scaled[col].values

print(f"\nDataFrame final creat:")
print(f"  Jugadors: {len(df_final)}")
print(f"  Columnes originals: {len(df_filtered.columns)}")
print(f"  Columnes escalades (noves): {len(numeric_features)}")
print(f"  Total columnes: {df_final.shape[1]}")

print(f"\nEstructura del DataFrame:")
print(f"  Identificadors: ['_id', 'player', 'n_records', 'total_min', 'avg_min']")
print(f"  Features originals: {len(numeric_features)} columnes")
print(f"  Features escalades: {len(numeric_features)} columnes amb prefix 'scaled_'")
print(f"  Indicador: ['n_outliers', 'best_zone']")

print(f"\nPrimeres columnes del dataset:")
print(df_final.columns.tolist()[:15])


1.8.1: CREAR DATAFRAME FINAL (ORIGINAL + ESCALAT)

DataFrame final creat:
  Jugadors: 98
  Columnes originals: 39
  Columnes escalades (noves): 30
  Total columnes: 69

Estructura del DataFrame:
  Identificadors: ['_id', 'player', 'n_records', 'total_min', 'avg_min']
  Features originals: 30 columnes
  Features escalades: 30 columnes amb prefix 'scaled_'
  Indicador: ['n_outliers', 'best_zone']

Primeres columnes del dataset:
['_id', 'n_records', 'total_min', 'avg_min', 'pts', 'fgm', 'fga', '3pm', '3pa', 'ftm', 'fta', 'oreb', 'dreb', 'ast', 'tov']


In [ ]:
# 1.8.2: Documentar cada columna del dataset
print("\n" + "=" * 80)
print("1.8.2: DOCUMENTACIÓ DE COLUMNES")
print("=" * 80)

# Crear diccionari de descripció per columna
column_descriptions = {
    # Identificadors
    '_id': 'MongoDB ID (mantingut per trazabilitat)',
    'player': 'Identificador del jugador (player_number o ID)',
    'n_records': 'Nombre de documents (partits) per a aquest jugador',
    'total_min': 'Total de minuts jugats (agregat)',
    'avg_min': 'Mitjana de minuts per partit',
    'n_outliers': 'Nombre de features que són outliers (IQR method)',
    'best_zone': 'Millor zona de tir per eficiència',
    
    # Features originals (estàdistiques agregades per jugador)
    'pts': 'Punts anotats (mitjana per partit)',
    'fgm': 'Tirs de camp encertats (mitjana)',
    'fga': 'Tirs de camp intentats (mitjana)',
    '3pm': 'Triples encertats (mitjana)',
    '3pa': 'Triples intentats (mitjana)',
    'ftm': 'Tirs lliures encertats (mitjana)',
    'fta': 'Tirs lliures intentats (mitjana)',
    'oreb': 'Rebots ofensius (mitjana)',
    'dreb': 'Rebots defensius (mitjana)',
    'ast': 'Assistències (mitjana)',
    'tov': 'Pèrdues de pilota (mitjana)',
    'stl': 'Robatoris (mitjana)',
    'blk': 'Tapons (mitjana)',
    'pf': 'Faltes personals (mitjana)',
    
    # Features derivades
    'possessions': 'Possessions estimades (FGA + 0.4*FTA + TOV)',
    'OER': 'Offensive Efficiency Rating (pts per 100 possessions)',
    'DER': 'Defensive Efficiency Rating (proxy per accions defensives per 100 possessions)',
    'pct_3pa': 'Percentatge de triples intentats (% sobre total FGA)',
    'pct_2pa': 'Percentatge de dobles intentats (% sobre total FGA)',
    'fg2_pct': 'Eficiència de triples (2PA% / 2PA)',
    'fg3_pct': 'Eficiència de triples (3PM% / 3PA)',
    'defensive_contribution': 'Contribució defensiva (STL + BLK)',
    'total_defensive_actions': 'Accions defensives totals (STL + BLK + TOV)',
    'best_zone_efficiency': 'Eficiència de la millor zona de tir',
    
    # Features escalades (totes amb prefix 'scaled_')
    'scaled_*': 'Versions estandarditzades (mean=0, std=1) de les features numèriques per a clustering K-Means/DBSCAN',
}

print(f"\nTotal de columnes documentades: {len(df_final.columns)}\n")

# Mostrar documentació categoritzada
categories = {
    'Identificadors': ['_id', 'player', 'n_records', 'total_min', 'avg_min'],
    'Qualitat de dades': ['n_outliers', 'best_zone'],
    'Features originals (14)': [c for c in numeric_features if c in df_final.columns and not c.startswith('scaled_')],
    'Features derivades (10)': ['possessions', 'OER', 'DER', 'pct_3pa', 'pct_2pa', 'fg2_pct', 'fg3_pct', 'defensive_contribution', 'total_defensive_actions', 'best_zone_efficiency'],
    'Features escalades': [f'scaled_{col}' for col in numeric_features]
}

for category, cols in categories.items():
    if cols:
        print(f"\n{category}:")
        for col in cols[:10]:  # Limitar a 10 per categoria per brevitat
            if col in column_descriptions:
                print(f"  {col:35s} - {column_descriptions[col]}")
        if len(cols) > 10:
            print(f"  ... i {len(cols) - 10} més")

print(f"\nDocumentació disponible per a totes les columnes")



1.8.2: DOCUMENTACIÓ DE COLUMNES

Total de columnes documentades: 69


Identificadors:
  _id                                 - MongoDB ID (mantingut per trazabilitat)
  player                              - Identificador del jugador (player_number o ID)
  n_records                           - Nombre de documents (partits) per a aquest jugador
  total_min                           - Total de minuts jugats (agregat)
  avg_min                             - Mitjana de minuts per partit

Qualitat de dades:
  n_outliers                          - Nombre de features que són outliers (IQR method)
  best_zone                           - Millor zona de tir per eficiència

Features originals (14):
  pts                                 - Punts anotats (mitjana per partit)
  fgm                                 - Tirs de camp encertats (mitjana)
  fga                                 - Tirs de camp intentats (mitjana)
  3pm                                 - Triples encertats (mitjana)
  3pa          

In [ ]:
# 1.8.3: Exportar dataset a CSV
print("\n" + "=" * 80)
print("1.8.3: EXPORTAR DATASET A CSV")
print("=" * 80)

import os

# Detectar carpeta de dades
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'notebooks':
    data_dir = os.path.dirname(current_dir)
else:
    data_dir = current_dir

data_dir = os.path.join(data_dir, 'data')
os.makedirs(data_dir, exist_ok=True)

# Rutes d'exportació
csv_original_path = os.path.join(data_dir, 'players_clustering_original.csv')
csv_scaled_path = os.path.join(data_dir, 'players_clustering_scaled.csv')

# Exportar dataset original
df_final.to_csv(csv_original_path, index=False, encoding='utf-8')
csv_original_size_kb = os.path.getsize(csv_original_path) / 1024

print(f"\nDataset original exportat: {csv_original_path}")
print(f"  Mida: {csv_original_size_kb:.1f} KB")
print(f"  Forma: {df_final.shape}")

# Exportar només features escalades (per a clustering directe)
df_scaled_only = df_final[['player'] + [f'scaled_{col}' for col in numeric_features]].copy()
df_scaled_only.to_csv(csv_scaled_path, index=False, encoding='utf-8')
csv_scaled_size_kb = os.path.getsize(csv_scaled_path) / 1024

print(f"\nDataset escalat exportat: {csv_scaled_path}")
print(f"  Mida: {csv_scaled_size_kb:.1f} KB")
print(f"  Columnes: 1 (player) + {len(numeric_features)} (scaled features)")
print(f"  Forma: {df_scaled_only.shape}")

print(f"\nNota: Usar 'players_clustering_scaled.csv' per a K-Means/DBSCAN clustering")
print(f"      Usar 'players_clustering_original.csv' per a interpretació posterior")



1.8.3: EXPORTAR DATASET A CSV

✓ Dataset original exportat: c:\Users\holaq\Desktop\FEB-Basketball-Clustering\data\players_clustering_original.csv
  Mida: 108.3 KB
  Forma: (98, 69)

✓ Dataset escalat exportat: c:\Users\holaq\Desktop\FEB-Basketball-Clustering\data\players_clustering_scaled.csv
  Mida: 55.8 KB
  Columnes: 1 (player) + 30 (scaled features)
  Forma: (98, 31)

Nota: Usar 'players_clustering_scaled.csv' per a K-Means/DBSCAN clustering
      Usar 'players_clustering_original.csv' per a interpretació posterior


In [ ]:
# 1.8.4: Resum final - Part 1 completada
print("\n" + "=" * 80)
print("1.8.4: RESUM FINAL - PART 1 (ETL/EDA/TRANSFORMACIONS) COMPLETADA")
print("=" * 80)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                        PART 1: ETL/EDA COMPLETADA                          ║
╚════════════════════════════════════════════════════════════════════════════╝

1. DADES ORIGINALS (MongoDB Atlas)
   • Col·leccions: 14 disponibles
   • Focus: FEB3_players_statistics (131,196 documents)
   • Jugadors únics: 110 (globals) → 98 retinguts (mostra >= 10 records)

2. EXPLORACIÓ INICIAL (Part 1.1-1.3)
   • Temporades: 4 (2020-2021 a 2023-2024)
   • Model seleccionat: JUGADORS (clustering de jugadors)
   • Agregació: GLOBAL (totes les temporades)
   • Filtre: minuts >= 100 per jugador

3. SELECCIÓ DE VARIABLES (Part 1.4)
   • Variables originals: 14 features
     - Ofensives: 7 (pts, fgm, fga, 3pm, 3pa, ftm, fta)
     - Rebot: 2 (oreb, dreb)
     - Teamwork: 1 (ast)
     - Defensives: 3 (tov, stl, blk)
     - Disciplina: 1 (pf)

4. NETEJA DE DADES (Part 1.5)
   • Valors nuls: Cap (EDA robust amb $avg)
   • Filtrat mostra: 110 → 98 jugadors (89.1% retenció)
   • Outliers: Detectats amb IQR, no eliminats

5. NOVES ESTADÍSTIQUES (Part 1.6)
   • Features derivades: 10
     - possessions (FGA + 0.4*FTA + TOV)
     - OER (pts/100 possessions)
     - DER (accions defensives/100 possessions, proxy)
     - Distribució 2 vs 3 (pct_2pa, pct_3pa, fg2_pct, fg3_pct)
     - Contribució defensiva (STL+BLK, STL+BLK+TOV)
     - Eficàcia per zona (best_zone_efficiency)
   • Total features: 14 + 10 = 24

6. NORMALITZACIÓ (Part 1.7)
   • StandardScaler aplicat: 30 features numèriques
   • Paràmetres: mean=0, std=1
   • Scaler guardat: data/models/scaler_clustering.pkl (1.5 KB)
   • Features llista: data/models/features_clustering.txt

7. DATASET FINAL (Part 1.8)
   • Jugadors: 98
   • Columnes: 69 totals
     - Identificadors: 5
     - Qualitat: 2
     - Features originals: 30
     - Features escalades: 30
     - Indicador: 2
   • Arxius exportats:
     - players_clustering_original.csv (amb totes les columnes)
     - players_clustering_scaled.csv (només features escalades)

════════════════════════════════════════════════════════════════════════════
                    PREPARACIÓ PER A PART 2: CLUSTERING
════════════════════════════════════════════════════════════════════════════

Fitxers disponibles per a Part 2:
data/players_clustering_scaled.csv → Ús directe per K-Means/DBSCAN
data/models/scaler_clustering.pkl → Transformació de futures dades
notebooks/01_connexio_mongo.ipynb → Codi complet ETL/EDA
""")

print(f"\nPART 1: ETL/EDA COMPLETADA SATISFACTÒRIAMENT")



1.8.4: RESUM FINAL - PART 1 (ETL/EDA/TRANSFORMACIONS) COMPLETADA

╔════════════════════════════════════════════════════════════════════════════╗
║                        PART 1: ETL/EDA COMPLETADA                          ║
╚════════════════════════════════════════════════════════════════════════════╝

1. DADES ORIGINALS (MongoDB Atlas)
   • Col·leccions: 14 disponibles
   • Focus: FEB3_players_statistics (131,196 documents)
   • Jugadors únics: 110 (globals) → 98 retinguts (mostra >= 10 records)

2. EXPLORACIÓ INICIAL (Part 1.1-1.3)
   • Temporades: 4 (2020-2021 a 2023-2024)
   • Model seleccionat: JUGADORS (clustering de jugadors)
   • Agregació: GLOBAL (totes les temporades)
   • Filtre: minuts >= 100 per jugador

3. SELECCIÓ DE VARIABLES (Part 1.4)
   • Variables originals: 14 features
     - Ofensives: 7 (pts, fgm, fga, 3pm, 3pa, ftm, fta)
     - Rebot: 2 (oreb, dreb)
     - Teamwork: 1 (ast)
     - Defensives: 3 (tov, stl, blk)
     - Disciplina: 1 (pf)

4. NETEJA DE DADES (Part